# BUSTAGO YOLO11 fine-tune — Colab fallback (R4)

**용도:** 로컬 RTX 3080 학습이 어려울 때 (CUDA 드라이버 문제, 메모리 부족 등) Google Colab T4에서 fine-tune 실행.

**근거 spec:** `docs/superpowers/specs/2026-05-16-yolo-data-track-design.md` §8 R4 폴백
**plan:** `docs/superpowers/plans/2026-05-16-yolo-data-track.md` Phase 4 / Phase 5

**사용 순서:**
1. 로컬에서 `datasets/bustago_person/self_v1/` 디렉토리를 zip으로 압축
2. Colab Runtime → Change runtime type → T4 GPU 선택
3. 이 노트북을 위에서부터 순서대로 실행
4. 학습 완료 후 `best.pt`를 로컬로 다운로드 → `runs/bustago/self_v1_finetune/weights/best.pt`에 배치
5. 로컬에서 `yolo val` 재실행해서 mAP 확인

**예상 시간:** T4 무료 GPU 기준 yolo11n / 100 epoch / 1000~3000장 → 1~3시간.

## 1. 환경 점검

In [ ]:
!nvidia-smi

↑ `Tesla T4` 또는 더 좋은 GPU가 보여야 함. CPU 모드면 Runtime → Change runtime type 다시 확인.

## 2. ultralytics 설치

In [ ]:
!pip install -q ultralytics==8.4.51

In [ ]:
import torch, ultralytics
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()} | device {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"ultralytics {ultralytics.__version__}")

## 3. 데이터셋 업로드

두 가지 방법 중 하나:

**방법 A — 직접 업로드 (간단, 데이터 ≤ 500MB 권장):**

In [ ]:
# 로컬에서 미리:
#   cd /home/ahble/projects/Capstone/bustago/datasets/bustago_person
#   zip -r self_v1.zip self_v1/images self_v1/labels
# 그 다음 아래 셀 실행 후 파일 선택 다이얼로그에서 self_v1.zip 업로드
from google.colab import files
uploaded = files.upload()  # self_v1.zip 선택

In [ ]:
import zipfile, pathlib
ZIP_PATH = next(iter(uploaded.keys()))  # 업로드된 zip 이름
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("datasets/bustago_person/self_v1")
print("Extracted to:", pathlib.Path("datasets/bustago_person/self_v1").resolve())
!ls datasets/bustago_person/self_v1/

**방법 B — Google Drive 마운트 (데이터 큰 경우, 또는 반복 사용):**

```python
from google.colab import drive
drive.mount('/content/drive')
# 데이터를 /content/drive/MyDrive/bustago/self_v1/ 에 미리 올려둠
!ln -s /content/drive/MyDrive/bustago/self_v1 datasets/bustago_person/self_v1
```

## 4. dataset yaml 작성

로컬의 `hardware/configs/bustago_person_self.yaml` 내용을 Colab에 직접 작성.

In [ ]:
YAML_CONTENT = '''path: datasets/bustago_person/self_v1
train: images/train
val: images/val
names:
  0: person
'''
import pathlib
pathlib.Path("bustago_person_self.yaml").write_text(YAML_CONTENT)
!cat bustago_person_self.yaml

## 5. fine-tune 1차 (Phase 4)

yolo11n 사전모델에서 시작. T4 메모리에 맞춰 imgsz=640, batch=16. 데이터셋 작으면 epochs 50~100.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(
    data='bustago_person_self.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    pretrained=True,
    cos_lr=True,
    close_mosaic=10,
    project='runs/bustago',
    name='self_v1_finetune',
    exist_ok=True,
)

## 6. self val 평가

In [ ]:
best = YOLO('runs/bustago/self_v1_finetune/weights/best.pt')
metrics = best.val(data='bustago_person_self.yaml', project='runs/bustago', name='self_v1_finetune_val', exist_ok=True)
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"P        : {metrics.box.mp:.4f}")
print(f"R        : {metrics.box.mr:.4f}")

이 4개 수치를 리포트 §4.1 fine-tune1 행에 기입.

## 7. (선택) 증강 fine-tune 2차 (Phase 5)

Phase 5에서는 hyperparam 추가.

In [ ]:
model2 = YOLO('yolo11n.pt')
results2 = model2.train(
    data='bustago_person_self.yaml',
    epochs=100, imgsz=640, batch=16, device=0,
    pretrained=True, cos_lr=True, close_mosaic=10,
    # 증강 hyperparam — Phase 5
    fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    mosaic=1.0,
    project='runs/bustago',
    name='self_v1_augmented',
    exist_ok=True,
)

In [ ]:
best2 = YOLO('runs/bustago/self_v1_augmented/weights/best.pt')
metrics2 = best2.val(data='bustago_person_self.yaml', project='runs/bustago', name='self_v1_augmented_val', exist_ok=True)
print(f"mAP50    : {metrics2.box.map50:.4f}")
print(f"mAP50-95 : {metrics2.box.map:.4f}")
print(f"P        : {metrics2.box.mp:.4f}")
print(f"R        : {metrics2.box.mr:.4f}")

## 8. 모델 다운로드 (로컬로)

In [ ]:
from google.colab import files
files.download('runs/bustago/self_v1_finetune/weights/best.pt')
# 필요하면 augmented도
# files.download('runs/bustago/self_v1_augmented/weights/best.pt')

로컬에서 받은 파일을:

```
mkdir -p runs/bustago/self_v1_finetune/weights
mv ~/Downloads/best.pt runs/bustago/self_v1_finetune/weights/best.pt
```

그 후 Phase 6 eval_counting.py에 모델 경로 인자로 넘기면 끝.

## 9. 참고

- **T4 메모리 부족 시:** `batch=8` 또는 `imgsz=480`으로 낮추기
- **세션 종료 위험:** Colab 무료 12시간 후 세션 종료. 학습 ≤ 3시간이면 안전
- **결과 보존:** runs/ 디렉토리는 세션 종료와 함께 사라짐 — 항상 best.pt를 Google Drive 또는 로컬로 즉시 다운로드
- **재현성:** seed 고정하려면 `model.train(..., seed=42)` 추가
- **데이터 크기 가이드:** 1000장 미만이면 epochs 100, 1000~3000장이면 epochs 50도 충분